Testing Eval with huggingface text-generation models

Using Langsmith

In [ ]:
from ragas.integrations.langchain import EvaluatorChain 
from ragas.metrics import faithfulness, context_precision
# from ragas.metrics.collections import faithfulness, context_precision
from langsmith.evaluation import LangChainStringEvaluator
from langchain_community.embeddings import HuggingFaceEmbeddings

In [5]:
from ..core.generation.llm import HUGGINGFACE
from ..core.utils.config import huggingface_config

resource = huggingface_config()

# llm
provider = HUGGINGFACE()
provider.model_name = resource['evaluation model']
provider.task = 'text-generation'
provider.load_model()
llm = provider.pipeline

# embedding model
embedding_model = HuggingFaceEmbeddings(model_name=resource['embedding model'])


ImportError: attempted relative import with no known parent package

Evaluation

Faithfullness

In [ ]:
faithfulness_chain = EvaluatorChain(
    metric = faithfulness, # the metric to use
    llm = llm, # or any other model
    embedding = embedding_model, # or any other embedding model
)

eval_result = faithfulness_chain(
    {
        "question": "What is the capital of France?",# user question
        "answer": "The capital of France is Paris.", # generated answer from the model
        "context": ["France is a country in Europe. Its capital city is Paris, known for its art, fashion, and culture."], # doc chunks available to the model
    }
)

print(eval_result)



using openeval

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

judge = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key="",
)
anthropic_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge = judge
    # judge=ChatAnthropic(model="claude-3-5-sonnet-latest", temperature=0.5),
)

In [7]:
inputs = "How is the weather in San Francisco?"
# These are fake outputs, in reality you would run your LLM-based system to get real outputs
outputs = "Thanks for asking! The current weather in San Francisco is sunny and 90 degrees."
# When calling an LLM-as-judge evaluator, parameters are formatted directly into the prompt
reference = "The weather in San Francisco is currently sunny and 90 degrees."
eval_result = anthropic_evaluator(
    inputs=inputs,
    outputs=outputs,
    reference_outputs=reference,


)

print(eval_result)

{'key': 'score', 'score': True, 'comment': 'The model output accurately and completely answers the question about the weather in San Francisco. It states that the weather is "sunny and 90 degrees," which aligns with the provided reference output. There are no factual errors, logical inconsistencies, or missing key information. The conversational opening "Thanks for asking!" does not detract from the correctness of the weather information provided. Thus, the score should be: true.', 'metadata': None}


Experimenting with store data function

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import List

def export_to_csv(path: str, filename: str, data: List[str]):
    """stores interaction data"""

    # ensuring that the main folder for storage exixts before proceeding
    folder = Path(path)
    if not folder.exists():
        raise ValueError('folder does not exist: copy the absolute path')
    try:
        # check if file exists or not inorder to prevent duplicate headers
        file = filename + ".csv"
        complete_file = Path(folder / file)
        file_exists = complete_file.exists()

        # convert the file to the preferred format and export for evaluation (.csv)
        interaction_data = np.array(data).reshape((1,3))
        df = pd.DataFrame(interaction_data,columns=['input','output','reference'])
        
        df.to_csv(complete_file, mode='a', header=not file_exists, index=False)
    except ValueError:
        raise ValueError('check the arguments')


check(
    r'C:\Users\Omolayo-Akinola\Documents\projects\engineering\RAG chatbot\data\raw',
    'test',
    ['test1','test2','test3']
)


check(
    r'C:\Users\Omolayo-Akinola\Documents\projects\engineering\RAG chatbot\data\raw',
    'test',
    ['test4','test5','test6']
)